In [ ]:
!pip install -q numpy==1.23.5 --force-reinstall
!pip install -q pandas matplotlib tqdm

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    IN_COLAB = False
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '.')

# データ拡張パイプラインをインポート
from src.pipelines import (
    PoseAugmentationPipeline,
    AugmentationPipelineConfig,
    AugmentationConfig,
    DataInputError,
    AugmentationError
)

print("✓ モジュールのインポートが完了しました")

In [ ]:
INPUT_CSV = 'data/detect/sample_video_11_01/player_classification_result.csv'
OUTPUT_CSV = 'output/augmented_pose_data.csv'
OUTPUT_METADATA = 'output/augmented_pose_data_metadata.json'
AUGMENTATION_FACTOR = 5
RANDOM_SEED = 42

print("=" * 70)
print("データ拡張パイプライン設定")
print("=" * 70)
print(f"入力CSV: {INPUT_CSV}")
print(f"出力CSV: {OUTPUT_CSV}")
print(f"メタデータ: {OUTPUT_METADATA}")
print(f"拡張倍率: {AUGMENTATION_FACTOR}x")
print(f"ランダムシード: {RANDOM_SEED}")
print("=" * 70)

In [ ]:
pipeline = PoseAugmentationPipeline.create_default(
    augmentation_factor=AUGMENTATION_FACTOR,
    random_seed=RANDOM_SEED
)

print("\n適用される拡張:")
print(f"  - 左右反転: {pipeline.config.augmentation.horizontal_flip} (確率 {pipeline.config.augmentation.horizontal_flip_prob})")
print(f"  - ガウシアンノイズ: {pipeline.config.augmentation.add_noise} (std {pipeline.config.augmentation.noise_std})")
print(f"  - 回転: {pipeline.config.augmentation.rotation} (±{pipeline.config.augmentation.rotation_range}度)")
print(f"  - スケーリング: {pipeline.config.augmentation.scaling}")
print(f"  - 関節ドロップアウト: {pipeline.config.augmentation.keypoint_dropout}")

In [ ]:
# データ拡張を実行
try:
    results = pipeline.augment_csv(
        input_csv=INPUT_CSV,
        output_csv=OUTPUT_CSV,
        output_metadata=OUTPUT_METADATA
    )
    
    print("\n" + "=" * 70)
    print("データ拡張の結果")
    print("=" * 70)
    print(f"元データ: {results['original_samples']} サンプル")
    print(f"拡張後: {results['augmented_samples']} サンプル")
    print(f"拡張倍率: {results['augmentation_factor']:.1f}x")
    print(f"出力CSV: {results['output_csv']}")
    print(f"メタデータ: {results['output_metadata']}")
    print("=" * 70)
    
except DataInputError as e:
    print(f"\n✗ データ読み込みエラー: {e}")
    print(f"  ファイルパス: {e.input_path}")
    print(f"  理由: {e.reason}")
except AugmentationError as e:
    print(f"\n✗ 拡張処理エラー: {e}")
except Exception as e:
    print(f"\n✗ 予期しないエラー: {e}")

In [ ]:
# 拡張データの確認
if Path(OUTPUT_CSV).exists():
    df = pd.read_csv(OUTPUT_CSV)
    
    print("\n" + "=" * 70)
    print("拡張データの統計")
    print("=" * 70)
    print(f"総行数: {len(df)}")
    print(f"列数: {len(df.columns)}")
    
    # augmentation_id別の統計
    if 'augmentation_id' in df.columns:
        print("\naugmentation_id別の分布:")
        aug_counts = df['augmentation_id'].value_counts().sort_index()
        for aug_id, count in aug_counts.items():
            label = "元データ" if aug_id == 0 else f"拡張データ {aug_id}"
            print(f"  {label}: {count} サンプル")
    
    # track_id別の統計
    if 'track_id' in df.columns:
        print("\ntrack_id別の分布:")
        track_counts = df['track_id'].value_counts().sort_index()
        for track_id, count in track_counts.items():
            print(f"  Track ID {track_id}: {count} サンプル")
    
    # データサンプルを表示
    print("\nデータサンプル（最初の5行）:")
    print(df[['frame', 'timestamp', 'track_id', 'augmentation_id']].head())
    
    print("=" * 70)
else:
    print(f"\n⚠ 出力ファイルが見つかりません: {OUTPUT_CSV}")